In [ ]:
# import libraries
from pathlib import Path
from getpass import getpass
import pandas as pd
import numpy as np

# specifically import binary psycopg
import os
os.environ["PSYCOPG_IMPL"] = "binary"
import psycopg

# setup path to csv file
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
path_raw_data = PROJECT_ROOT / "data" / "raw" / "PFAS Sample Sites - Surface Water and Fish Tissue.csv"

# read csv
df_locs = pd.read_csv(
    path_raw_data,
    usecols=['OBJECTID', 'wkt_geom_EPSG_3071', 'PRIMARY_STATION_NAME']
)

In [3]:
# get connection URI
connection_uri = getpass()

In [ ]:
# get ca file
ca_path = input()

In [ ]:
# create connection with database
conn = psycopg.connect(
    connection_uri,
    sslmode = 'verify-ca',
    sslrootcert = ca_path,
    connect_timeout = 10,
    gssencmode="disable"
)

In [ ]:
# test database connection
conn.execute(
    "SELECT COUNT(*) FROM public.sampling_locations"
).fetchone()

(0,)

In [11]:
# convert df to list of tuples
records = list(
    df_locs[
        ["OBJECTID", "wkt_geom_EPSG_3071", "PRIMARY_STATION_NAME"]
    ].itertuples(index=False, name=None)
)

In [12]:
# create sql statement to insert records
insert_sql = """
    INSERT INTO public.sampling_locations (
        objectid,
        geom,
        primary_station_name
    )
    VALUES (%s, ST_GeomFromText(%s, 3071), %s)
"""

In [13]:
# execute sql code
try:
    with conn.cursor() as cur:
        cur.executemany(insert_sql, records)

    conn.commit()
except Exception:
    conn.rollback()
    raise